In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # EX · IA aplicada a Trust & Safety
# MAGIC
# MAGIC ## YelpConnect
# MAGIC
# MAGIC ### Objetivo
# MAGIC
# MAGIC Enriquecer una muestra controlada de reseñas mediante
# MAGIC Databricks AI Functions.
# MAGIC
# MAGIC Se analizan:
# MAGIC
# MAGIC - Sentimiento.
# MAGIC - Tema principal.
# MAGIC - Inconsistencia entre rating y sentimiento.
# MAGIC
# MAGIC ## Importante
# MAGIC
# MAGIC Los resultados de IA constituyen señales analíticas.
# MAGIC
# MAGIC **No representan fraude confirmado.**
# MAGIC
# MAGIC Una inconsistencia solamente convierte una reseña
# MAGIC en candidata a análisis por Trust & Safety.
# MAGIC
# MAGIC ## FinOps
# MAGIC
# MAGIC Máximo: 500 reseñas.

# COMMAND ----------

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

GOLD_SCHEMA = "yelp_gold"

SOURCE_TABLE = (
    f"{CATALOG}."
    f"{GOLD_SCHEMA}."
    "gld_review_risk"
)

AI_TABLE = (
    f"{CATALOG}."
    f"{GOLD_SCHEMA}."
    "gld_review_ai_enrichment"
)

MAX_SAMPLE = 500

print("=" * 72)
print("YELPCONNECT - TRUST & SAFETY AI ENRICHMENT")
print("=" * 72)

print(
    f"Source : {SOURCE_TABLE}"
)

print(
    f"Target : {AI_TABLE}"
)

print(
    f"Sample : <= {MAX_SAMPLE}"
)

print("=" * 72)


# COMMAND ----------

# ============================================================
# 2. VALIDAR FUENTE
# ============================================================

if not spark.catalog.tableExists(
    SOURCE_TABLE
):

    raise Exception(
        f"""
        ❌ No existe:

        {SOURCE_TABLE}

        Ejecutar previamente
        el pipeline Gold.
        """
    )

print(
    "✔ Gold Review Risk disponible."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 3. Muestra determinística
# MAGIC
# MAGIC Se utilizan:
# MAGIC
# MAGIC - 250 reseñas que poseen señales T&S.
# MAGIC - 250 reseñas sin señales T&S.
# MAGIC
# MAGIC Esto permite comparar ambos grupos sin analizar
# MAGIC las 400,000 reseñas y consumir cuota innecesariamente.


# COMMAND ----------

# DBTITLE 1,Crear muestra IA
spark.sql(
    f"""
    CREATE OR REPLACE TEMP VIEW vw_ai_review_sample
    AS

    WITH con_signal AS
    (
        SELECT
            *,
            'WITH_TS_SIGNAL' AS ai_sample_group,

            ROW_NUMBER() OVER
            (
                ORDER BY
                    xxhash64(review_id)
            ) AS rn

        FROM
            {SOURCE_TABLE}

        WHERE
            signal_count > 0
    ),

    sin_signal AS
    (
        SELECT
            *,
            'WITHOUT_TS_SIGNAL' AS ai_sample_group,

            ROW_NUMBER() OVER
            (
                ORDER BY
                    xxhash64(review_id)
            ) AS rn

        FROM
            {SOURCE_TABLE}

        WHERE
            signal_count = 0
    )

    SELECT *
    FROM con_signal
    WHERE rn <= 250

    UNION ALL

    SELECT *
    FROM sin_signal
    WHERE rn <= 250
    """
)


sample_count = (
    spark.table(
        "vw_ai_review_sample"
    )
    .count()
)


print(
    f"✔ Reviews seleccionadas: "
    f"{sample_count}"
)


if sample_count > MAX_SAMPLE:

    raise Exception(
        """
        ❌ La muestra supera
        el máximo permitido.
        """
    )


# COMMAND ----------

display(
    spark.sql(
        """
        SELECT

            ai_sample_group,

            COUNT(*) AS reviews

        FROM
            vw_ai_review_sample

        GROUP BY
            ai_sample_group
        """
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 4. Enriquecimiento con IA
# MAGIC
# MAGIC Se utilizan funciones de IA para generar:
# MAGIC
# MAGIC - sentimiento
# MAGIC - tema
# MAGIC
# MAGIC El texto de la reseña se considera entrada no estructurada.


# COMMAND ----------

# DBTITLE 1,AI Sentiment + Classification
spark.sql(
    f"""
    CREATE OR REPLACE TABLE {AI_TABLE}

    COMMENT
    'Muestra gobernada de reseñas YelpConnect enriquecidas mediante IA. '
    'Contiene sentimiento, tema e inconsistencia rating-sentimiento. '
    'Uso: análisis Trust & Safety. '
    'Las señales no representan fraude confirmado.'

    AS

    WITH ai_base AS
    (
        SELECT

            review_id,
            business_id,
            user_id,

            stars,

            review_text,

            ai_sample_group,

            signal_count,
            signal_level,
            investigation_priority,
            requires_investigation,

            evaluation_run_id,

            ai_analyze_sentiment(
                review_text
            ) AS ai_sentiment,

            ai_classify(
                review_text,

                ARRAY(
                    'comida',
                    'servicio',
                    'precio',
                    'ambiente',
                    'producto',
                    'experiencia_general',
                    'otro'
                )
            ) AS ai_topic

        FROM
            vw_ai_review_sample
    )

    SELECT

        *,

        CASE

            WHEN
                stars >= 4
                AND LOWER(ai_sentiment) = 'negative'

                THEN TRUE

            WHEN
                stars <= 2
                AND LOWER(ai_sentiment) = 'positive'

                THEN TRUE

            ELSE FALSE

        END AS rating_sentiment_inconsistent,

        current_timestamp()
            AS ai_processed_at,

        'Databricks AI Functions'
            AS ai_engine,

        'ai_analyze_sentiment'
            AS ai_sentiment_function,

        'ai_classify'
            AS ai_topic_function

    FROM
        ai_base
    """
)


print(
    f"✔ Tabla IA creada: "
    f"{AI_TABLE}"
)


# COMMAND ----------

# ============================================================
# 5. VALIDACIÓN
# ============================================================

ai_count = (
    spark.table(
        AI_TABLE
    )
    .count()
)


if ai_count > MAX_SAMPLE:

    raise Exception(
        f"""
        ❌ FinOps Gate.

        La tabla IA contiene:

        {ai_count}

        Máximo permitido:

        {MAX_SAMPLE}
        """
    )


print(
    f"✔ Reviews enriquecidas: "
    f"{ai_count}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 6. Inconsistencia rating ↔ sentimiento
# MAGIC
# MAGIC Esta es una señal heurística adicional para
# MAGIC Trust & Safety.
# MAGIC
# MAGIC No representa fraude confirmado.


# COMMAND ----------

display(
    spark.sql(
        f"""
        SELECT

            stars,

            ai_sentiment,

            COUNT(*) AS reviews

        FROM
            {AI_TABLE}

        GROUP BY

            stars,
            ai_sentiment

        ORDER BY

            stars,
            ai_sentiment
        """
    )
)


# COMMAND ----------

# ============================================================
# 7. CANDIDATAS POR INCONSISTENCIA
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            review_id,

            stars,

            ai_sentiment,

            ai_topic,

            signal_count,

            signal_level,

            investigation_priority,

            requires_investigation,

            review_text

        FROM
            {AI_TABLE}

        WHERE
            rating_sentiment_inconsistent = TRUE

        ORDER BY

            signal_count DESC,

            stars DESC
        """
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 8. IA vs señales T&S actuales


# COMMAND ----------

display(
    spark.sql(
        f"""
        SELECT

            ai_sample_group,

            COUNT(*) AS reviews,

            SUM(
                CASE

                    WHEN
                        rating_sentiment_inconsistent
                        = TRUE

                    THEN 1

                    ELSE 0

                END
            ) AS inconsistentes,

            ROUND(
                100.0
                *
                SUM(
                    CASE

                        WHEN
                            rating_sentiment_inconsistent
                            = TRUE

                        THEN 1

                        ELSE 0

                    END
                )

                /

                COUNT(*),

                2
            ) AS pct_inconsistentes

        FROM
            {AI_TABLE}

        GROUP BY
            ai_sample_group
        """
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 9. Nuevas candidatas que las reglas actuales no detectaron


# COMMAND ----------

display(
    spark.sql(
        f"""
        SELECT

            review_id,

            stars,

            ai_sentiment,

            ai_topic,

            review_text

        FROM
            {AI_TABLE}

        WHERE
            rating_sentiment_inconsistent = TRUE

            AND signal_count = 0

        ORDER BY
            review_id
        """
    )
)


# COMMAND ----------

# ============================================================
# 10. KPIs IA
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            COUNT(*)
                AS reviews_analizadas,

            SUM(
                CASE

                    WHEN rating_sentiment_inconsistent

                    THEN 1

                    ELSE 0

                END
            )
                AS inconsistencias_rating_sentimiento,

            SUM(
                CASE

                    WHEN
                        rating_sentiment_inconsistent
                        AND signal_count = 0

                    THEN 1

                    ELSE 0

                END
            )
                AS nuevas_candidatas_ai,

            SUM(
                CASE

                    WHEN
                        rating_sentiment_inconsistent
                        AND signal_count > 0

                    THEN 1

                    ELSE 0

                END
            )
                AS coincidencias_ai_ts

        FROM
            {AI_TABLE}
        """
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 11. Gobierno de IA
# MAGIC
# MAGIC ## Interpretación
# MAGIC
# MAGIC El resultado generado por las funciones de IA se considera
# MAGIC **dato derivado**, no una verdad de negocio.
# MAGIC
# MAGIC Un sentimiento negativo o una inconsistencia entre rating
# MAGIC y sentimiento no demuestra fraude.
# MAGIC
# MAGIC Se utiliza únicamente como señal adicional para priorizar
# MAGIC análisis humanos.
# MAGIC
# MAGIC ## Riesgos
# MAGIC
# MAGIC Persistir salidas de un modelo introduce riesgos adicionales:
# MAGIC
# MAGIC - clasificación incorrecta;
# MAGIC - sesgo del modelo;
# MAGIC - comportamiento no determinístico;
# MAGIC - cambios de comportamiento ante nuevas versiones del modelo;
# MAGIC - interpretación incorrecta de ironía o sarcasmo;
# MAGIC - exposición de información sensible contenida en texto libre;
# MAGIC - dependencia tecnológica del proveedor/modelo;
# MAGIC - pérdida de trazabilidad si no se registra cómo se obtuvo
# MAGIC   cada resultado.
# MAGIC
# MAGIC ## Controles
# MAGIC
# MAGIC YelpConnect establece:
# MAGIC
# MAGIC 1. Las salidas IA son señales, no decisiones automáticas.
# MAGIC 2. La decisión final permanece bajo revisión humana.
# MAGIC 3. Las salidas se persisten en una tabla gobernada por
# MAGIC    Unity Catalog.
# MAGIC 4. Se registra fecha de procesamiento y función utilizada.
# MAGIC 5. El procesamiento se limita a una muestra controlada por
# MAGIC    FinOps.
# MAGIC 6. El resultado puede compararse contra el risk score
# MAGIC    determinístico existente.
# MAGIC 7. Una nueva regla T&S basada en IA debe pasar por
# MAGIC    experimentación, versionado y aprobación antes de
# MAGIC    incorporarse al score operativo.


# COMMAND ----------

# ============================================================
# 12. RESULTADO FINAL
# ============================================================

print()
print("=" * 72)
print("AI TRUST & SAFETY ENRICHMENT FINALIZADO")
print("=" * 72)

print(
    f"Reviews analizadas : "
    f"{ai_count}"
)

print(
    f"Tabla              : "
    f"{AI_TABLE}"
)

print()
print(
    "Las salidas IA son señales analíticas."
)

print(
    "NO representan fraude confirmado."
)

print("=" * 72)